# ProteinGym

In [1]:
# conda create --name proteingym_env --file environments/proteingym_env.txt -c conda-forge -c bioconda -c pytorch
# !pip install proteingym -y

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/grid/koo/home/schilder/projects/GenomeEncoder/')
import pandas as pd
import src.haplosaurus as hs
import src.ESM as ESM
import src.utils as utils
import src.biomart as bm
import sys

# import proteingym as pg # Pip version
sys.path.append('ProteinGym')
import proteingym.utils.download as pgd # Local version


## Download data

In [2]:
resources_df = pgd.get_resources_df()

In [3]:
pg_resources = pgd.download_resources(
    resources_df.loc[resources_df['Filename'].isin(
        ['clinical_ProteinGym_substitutions.zip', 
         'clinical_ProteinGym_indels.zip']
        )],
    error=False)

In [4]:
pgd.count_resources(pg_resources)

{'clinical_ProteinGym_substitutions': '2525 file(s)',
 'clinical_ProteinGym_indels': '1555 file(s)'}

In [5]:
proteins_df = pd.DataFrame(
    list(set([
    os.path.basename(f).replace(".csv", "") 
    for key in ['clinical_ProteinGym_substitutions', 'clinical_ProteinGym_indels']
    for f in pg_resources[key]
])),
    columns=['protein']
)
proteins_df['RefSeq peptide ID'] = proteins_df['protein'].str.split('.').str[0]

id_map = bm.get_id_map()
proteins_df = bm.map_ids(proteins_df, id_map=id_map)
proteins_df


Loading from cache ==> /grid/koo/home/schilder/.cache/biomart/id_map.csv.gz


,protein,RefSeq peptide ID,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Protein stable ID,Protein stable ID version,RefSeq peptide predicted ID
0,NP_056989.2,NP_056989,ENSG00000122779,ENSG00000122779.18,ENST00000343526,ENST00000343526.9,ENSP00000340507,ENSP00000340507.4,XP_024302749
1,NP_056989.2,NP_056989,ENSG00000122779,ENSG00000122779.18,ENST00000343526,ENST00000343526.9,ENSP00000340507,ENSP00000340507.4,XP_054215260
2,NP_055454.1,NP_055454,ENSG00000198677,ENSG00000198677.13,ENST00000358746,ENST00000358746.7,ENSP00000351596,ENSP00000351596.3,XP_047273893
3,NP_997244.4,NP_997244,ENSG00000150893,ENSG00000150893.11,ENST00000280481,ENST00000280481.9,ENSP00000280481,ENSP00000280481.7,NaN
4,NP_001104262.1,NP_001104262,ENSG00000169057,ENSG00000169057.26,ENST00000453960,ENST00000453960.7,ENSP00000395535,ENSP00000395535.2,XP_047298078
...,...,...,...,...,...,...,...,...,...
8839,NP_000477.1,NP_000477,ENSG00000167580,ENSG00000167580.8,ENST00000199280,ENST00000199280.4,ENSP00000199280,ENSP00000199280.3,NaN
8840,UPI0001838820,UPI0001838820,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8841,NP_006252.4,NP_006252,ENSG00000280635,ENSG00000280635.1,ENST00000629850,ENST00000629850.1,ENSP00000486962,ENSP00000486962.1,NaN
8842,NP_006252.4,NP_006252,ENSG00000274382,ENSG00000274382.2,ENST00000614006,ENST00000614006.2,ENSP00000484677,ENSP00000484677.1,NaN


## Count variants

### Substitutions

In [ ]:
clinical_subs_df = pgd.concat_csvs(
    pg_resources, 
    key='clinical_ProteinGym_substitutions',
    )

In [21]:
# Report stats
haplotypes_per_protein = 37.5
print(clinical_subs_df['protein'].nunique()*haplotypes_per_protein,
      "haplotypes across",len(clinical_subs_df), 
      "substitutions in",clinical_subs_df['protein'].nunique(),"proteins")

sum(clinical_subs_df.groupby('mutant').size()*haplotypes_per_protein)

94687.5 haplotypes across 62727 substitutions in 2525 proteins


2352262.5

In [ ]:
# Using all benign/pathogenic variants per protein
print(sum(clinical_subs_df.groupby('protein').size()*haplotypes_per_protein))
# Using only 1 benign/pathogenic variant per protein
print(clinical_subs_df['protein'].nunique()*2*haplotypes_per_protein)

2352262.5
189375.0


### Indels

In [ ]:
clinical_indels_df = pgd.concat_csvs(
    pg_resources, 
    fkey='clinical_ProteinGym_indels')

Reading files in clinical_ProteinGym_indels:   0%|          | 0/1555 [00:00<?, ?it/s]

In [14]:
# Report stats
haplotypes_per_protein = 37.5
print(clinical_indels_df['refseq_unique_id'].nunique()*haplotypes_per_protein,
      "estimated haplotypes across",len(clinical_indels_df), 
      "substitutions in",clinical_indels_df['refseq_unique_id'].nunique(),"proteins")
print("Total estimated haplotype sequences:",
      clinical_indels_df['refseq_unique_id'].nunique()*haplotypes_per_protein)

32175.0 haplotype sequences across 2878 substitutions in 858 proteins


In [ ]:
# Using all benign/pathogenic variants per protein
print(sum(clinical_indels_df.groupby('refseq_unique_id').size()*haplotypes_per_protein))
# Using only 1 benign/pathogenic variant per protein
print(clinical_indels_df['refseq_unique_id'].nunique()*2*haplotypes_per_protein)

76462.5
64350.0


## Benchmarking

In [6]:
# import sys
# sys.path.append('ProteinGym')
import src.predict_esm as ESMp

In [7]:
ESM.list_models()

- esm
- esm1_t12_85M_UR50S
- esm1_t34_670M_UR100
- esm1_t34_670M_UR50D
- esm1_t34_670M_UR50S
- esm1_t6_43M_UR50S
- esm1b_t33_650M_UR50S
- esm1v_t33_650M_UR90S
- esm1v_t33_650M_UR90S_1
- esm1v_t33_650M_UR90S_2
- esm1v_t33_650M_UR90S_3
- esm1v_t33_650M_UR90S_4
- esm1v_t33_650M_UR90S_5
- esm2_t12_35M_UR50D
- esm2_t30_150M_UR50D
- esm2_t33_650M_UR50D
- esm2_t36_3B_UR50D
- esm2_t48_15B_UR50D
- esm2_t6_8M_UR50D
- esm_if1_gvp4_t16_142M_UR50
- esm_msa1_t12_100M_UR50S
- esm_msa1b_t12_100M_UR50S
- esmfold_v0
- esmfold_v1


In [8]:
tx_ids = hs.list_haplotypes()

Found 403 haplotypes in /grid/koo/home/schilder/projects/data/haplosaurus/haplotypes/


In [9]:
haplotypes = hs.get_haplotypes(tx_ids=tx_ids[:50], 
                               cache_only=True)

Getting haplotypes:   0%|          | 0/50 [00:00<?, ?it/s]

In [10]:
hap_seqs = hs.get_haplotype_seqs(haplotypes, 
                                 aligned=True, 
                                 use_protein_ids=True, 
                                 add_haplotype_names=True)

Getting haplotype sequences:   0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [12]:
# hap_seqs

In [13]:

# Get a list of usable proteins
id_df = proteins_df.loc[proteins_df['Protein stable ID'].isin(hap_seqs.keys())]
protein_ids = [os.path.basename(x).replace(".csv", "") for x in pg_resources['clinical_ProteinGym_substitutions']]
protein_ids = utils.intersect(protein_ids, id_df.protein)
len(protein_ids)
    



18

In [ ]:
# Script to generate mutational landscape
# predict_esm.py script from: https://github.com/facebookresearch/esm/blob/main/examples/variant-prediction/predict.py
import os
from tqdm.auto import tqdm
save_dir = '/grid/koo/home/schilder/projects/data/1KG/vep/'
force = False

# Scoring strategy
scoring_strategy = ["wt-marginals", "masked-marginals", "pseudo-ppl"][2]

# Model location
model_location = "esm1v_t33_650M_UR90S_1"

save_paths = {}
# Iterate over proteins
for pid in tqdm(protein_ids[:3], desc="Processing proteins"):
    # Map protein id to ensp_id
    ensp_id = id_df.loc[id_df['protein']==pid]['Protein stable ID'].tolist()[0]
    assert len(ensp_id)>0
    # Load clinical data
    variants_path = [x for x in pg_resources['clinical_ProteinGym_substitutions'] if pid in x][0]
    assert len(variants_path)>0
    # Iterate over haplotypes
    for seq_name, seq in tqdm(hap_seqs[ensp_id], 
                              desc="Processing haplotypes", 
                              leave=False):
        is_ref = seq_name.endswith('REF')
        # create save path
        save_path = os.path.join(save_dir, 
                                 model_location, 
                                 pid, 
                                 scoring_strategy,
                                 f"{seq_name}.csv")
        save_paths[(pid, seq_name, scoring_strategy)] = save_path
        if os.path.exists(save_path) and not force:
            continue
        else:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        # Run script via command line
        ESMp.main(
            dms_input=variants_path,
            dms_output=save_path,
            model_location=[model_location],
            sequence=seq,
            mutation_col="mutant",
            offset_idx=1,
            scoring_strategy=scoring_strategy,
            is_ref=is_ref
        )
        # !python src/predict_esm.py \
        #     --model-location {model_location} \
        #     --sequence {seq} \
        #     --dms-input {variants_path} \
        #     --mutation-col mutant \
        #     --dms-output {save_path} \
        #     --offset-idx 1 \
        #     --scoring-strategy {scoring_strategy} 
        #     # --msa-path ./data/Encapsidation_protein_22K.a3m

Processing proteins:   0%|          | 0/3 [00:00<?, ?it/s]

Processing haplotypes:   0%|          | 0/64 [00:00<?, ?it/s]

/grid/koo/home/schilder/.local/lib/python3.9/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


Transferred model to GPU




  0%|          | 0/6 [00:00<?, ?it/s]

 33%|███▎      | 2/6 [01:19<02:38, 39.56s/it]